# Customer Churn Prediction
## By Kosisochukwu Caleb Nwatu

***This project is aimed towards helping telecom bussiness increase customer retention by providing***

**1.0 Import libries:**

In [750]:
import numpy as np
import pandas as pd
import kagglehub
import joblib
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

# Make results the same every time we run the notebook
np.random.seed(42)

**2.0 Load Dataset:**

In [751]:
# # Download latest version
# path = kagglehub.dataset_download("blastchar/telco-customer-churn")

# print("Path to dataset files:", path)
path =  (r'C:\Users\USER\.cache\kagglehub\datasets\blastchar\telco-customer-churn\versions\1')
#Read Dataset
df = pd.read_csv(f'{path}/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print('\nFirat few rows of the dataset')
df.head()


Firat few rows of the dataset


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


**3.0 Exploratory Data Analysis (EDA):**

In [752]:
# How many rows and columns does this dataset contain
print(f'The dataset has {df.shape[0]} rows and {df.shape[1]} columns')

The dataset has 7043 rows and 21 columns


In [753]:
# Getting information on the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


Cross check the data type of each column to with the content of each from the few rows we displayed previously. The TotalCharges column has an incorrect data type, it will need conversion to the best data type that describes it's content.


In [754]:
# Convert data type
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'], errors='coerce'
)

In [755]:
# Checking convertion status
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [756]:
# Check the statistics. 
# The include='all' arrgument can be passed to include non-numerical columns
df.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7032.000000
mean,0.162147,32.371149,64.761692,2283.300441
std,0.368612,24.559481,30.090047,2266.771362
min,0.000000,0.000000,18.250000,18.800000
25%,0.000000,9.000000,35.500000,401.450000
50%,0.000000,29.000000,70.350000,1397.475000
75%,0.000000,55.000000,89.850000,3794.737500
max,1.000000,72.000000,118.750000,8684.800000


In [757]:
# Fix columns with object data type 
data = {}

for col in df.columns:
    if df[col].dtype == 'object':
        data[col] = (
            df[col]
            .str.strip()
            .str.replace(r'\s+', " ", regex=True)
            .str.title()
            .replace(" ", np.nan)
        )
    else:
        data[col] = df[col]

df = pd.DataFrame(data)

# Display first few rows to confirm changes
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-Vhveg,Female,0,Yes,No,1,No,No Phone Service,Dsl,No,...,No,No,No,No,Month-To-Month,Yes,Electronic Check,29.85,29.85,No
1,5575-Gnvde,Male,0,No,No,34,Yes,No,Dsl,Yes,...,Yes,No,No,No,One Year,No,Mailed Check,56.95,1889.50,No
2,3668-Qpybk,Male,0,No,No,2,Yes,No,Dsl,Yes,...,No,No,No,No,Month-To-Month,Yes,Mailed Check,53.85,108.15,Yes
3,7795-Cfocw,Male,0,No,No,45,No,No Phone Service,Dsl,Yes,...,Yes,Yes,No,No,One Year,No,Bank Transfer (Automatic),42.30,1840.75,No
4,9237-Hqitu,Female,0,No,No,2,Yes,No,Fiber Optic,No,...,No,No,No,No,Month-To-Month,Yes,Electronic Check,70.70,151.65,Yes


To ensure we work with a clean dataset, the **For loop** was used. The loop strips each column with datatype string of extra scpaces while formating cells with only " " to NaN

In [758]:
# Details on dataset's missing values
print(f'Null values per column in a list:\n {df.isnull().sum().to_list()}') 
print('\nNull per column in dataframe:')
df.isnull().sum()

Null values per column in a list:
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 11, 0]

Null per column in dataframe:


customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [759]:
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
print('TotalCharges NAN has beeb filled with the columns median.')

TotalCharges NAN has beeb filled with the columns median.


In [760]:
# Confirming that NaN was filled
print(f'Number of null values found: {df.isnull().sum().sum()}')

Number of null values found: 0


In [761]:
# Check & drop duplicate
print(f'Number of duplicates found: {df.duplicated().sum()}') # Check
df.drop_duplicates(inplace=True) # Drop

Number of duplicates found: 0


**4.0 Feautures selection**

In [762]:
X = df.iloc[:, 1:-1]
y = df.iloc[:,-1]

print(X.shape)
print(y.shape)

(7043, 19)
(7043,)


**5.0 Categorical label Encoding**

In [763]:
# Label_encoding
le_col = [
    'gender', 
    'Partner', 
    'Dependents', 
    'PhoneService', 
    'PaperlessBilling'
]

le = LabelEncoder()
X[le_col] = X[le_col].apply(le.fit_transform)


**6.0 Data Spliting:**

In [764]:
# Split into traning and testing set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y   # stratify keeps the churn Yes/No ratio equal in both sets
)

print('X_train: ',X_train.shape)
print('X_test: ',X_test.shape)
print('y_train: ',y_train.shape)
print('y_test: ',y_test.shape)

X_train:  (5634, 19)
X_test:  (1409, 19)
y_train:  (5634,)
y_test:  (1409,)


**7.0 Categorical Encoding and Scaling**

In [765]:

# Encoding & Data Scaling
scale_cols = [
    'SeniorCitizen', 
    'tenure', 
    'MonthlyCharges', 
    'TotalCharges'
]# Standard_scaling

oh_cols = [
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaymentMethod'
]# Onehot_encoding

ct = ColumnTransformer(
    transformers=[
        ("onehot", OneHotEncoder(drop="first"), oh_cols),
        ("scale", StandardScaler(), scale_cols)
    ], 
    remainder="passthrough"
)



**8.0 Model traning:**

In [766]:
logi = make_pipeline(ct ,LogisticRegression())
logi.fit(X_train, y_train)


,steps,"[('columntransformer', ...), ('logisticregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehot', ...), ('scale', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [767]:
y_pred = logi.predict(X_test)
print("Logistic Regression accuracy:", round(accuracy_score(y_test, y_pred), 3))

Logistic Regression accuracy: 0.806


**7.0 Save model**

In [768]:
# Save the Model
with open('Customer_Churn_model.pkl', 'wb') as f:
    pickle.dump(logi, f)

print('Logi saved as pkl')


Logi saved as pkl
